# Reproducing Mohebi et al. (2019) from CRCNS vta-2

**Paper.** Mohebi A, Pettibone JR, Hamid AA, Wong JT, Vinson LT, Patriarchi T, Tian L, Kennedy RT,
Berke JD (2019). Dissociable dopamine dynamics for learning and motivation. *Nature* 570, 65-70.
[doi:10.1038/s41586-019-1235-y](https://doi.org/10.1038/s41586-019-1235-y). The claims file
(`claims/mohebi-2019.yaml`) was written from the version of record as served by Europe PMC
(PMC6555489), with the Methods, the Extended Data legends and the Supplementary Figure; the legends
of main Figures 1 to 5 were not in that text.

**Data set.** CRCNS vta-2, Mohebi & Berke (2019), "Spiking activity of dopaminergic cells in VTA-l
during multitude of tasks and conditions", [doi:10.6080/K0VQ30V9](https://doi.org/10.6080/K0VQ30V9),
streamed from the public bucket `crcnsarchive` (`vta-2/live/data_and_code.zip`).

**Version.** CRCNS data sets are not versioned, so the version is recorded as `unversioned`. The
data set has one published state: its only data file, `data_and_code.zip`, whose MD5 in the data
set's `checksums.md5` is `693383b76465b128cb3190591ef43d2d` (the data check below confirms that
the bucket serves this content), with the data description version 0.6 (5 August 2019). There is
no earlier or later version to choose from (nothing under `vta-2/old/`), and the paper cites the
data set by its DOI only.

**What the data set holds.** Only the spike times of the 29 optogenetically identified dopamine
cells (27 from four rats with VTA-l virus, 2 from the rat with NAc core virus), with the bandit,
Pavlovian, laser and sleep segments of each recording and their behavioural events. It holds no
microdialysis, photometry, voltammetry, histology or in vitro data, and none of the untagged VTA-l
units. The claims on those data are blocked (`data_missing`).

**Code in the data set.** The zip also holds three MATLAB files by the authors (`pelletpermin.m`,
`getFiringRate.m`, `Fig5_replicate.m`). They are read here as method sources where the paper is
silent (the Gaussian kernel's SD, and how `pelletpermin.m` counts recent rewards); each use is
cited in the helper `vta.py`.

| Claim | Tested here |
|---|---|
| fig1b-latency-scales-with-reward-rate | tested |
| ed1b-latency-depends-on-recent-outcomes | blocked, data_missing (microdialysis sessions) |
| fig1cd-nac-core-dopamine-reward-rate | blocked, data_missing (microdialysis) |
| fig1ce-ventral-prelimbic-dopamine-reward-rate | blocked, data_missing (microdialysis) |
| ed1-behavior-similar-across-subregions | blocked, data_missing (microdialysis sessions) |
| fig2-tagged-dopamine-cells | tested (on the deposited cells only) |
| fig2d-dopamine-firing-rates | tested |
| fig2d-dopamine-lower-rate-than-other-vta | blocked, data_missing (untagged VTA-l units) |
| fig2d-dopamine-longer-waveforms | blocked, data_missing (untagged VTA-l units) |
| fig2d-non-dopamine-cluster | blocked, data_missing (untagged VTA-l units) |
| ed5-pavlovian-dopamine-click-responses | tested |
| ed5-pavlovian-dopamine-cue-responses | tested |
| ed5a-pavlovian-behavior | tested |
| fig2efgh-tonic-firing-indifferent-to-reward-rate | tested |
| fig2fh-non-dopamine-no-tonic-change | blocked, data_missing (untagged VTA-l units) |
| fig2i-burst-rate-no-change | tested |
| fig2i-no-inactive-states | tested |
| ed3-ctb-retrograde-labelling | blocked, data_missing (histology) |
| retro-tagged-cells-resemble-others | tested |
| fig3a-dlight-specific | blocked, data_missing (in vitro imaging) |
| ed5-dlight-pavlovian-responses | blocked, data_missing (photometry) |
| fig3cd-dlight-reward-rate-minutes | blocked, data_missing (photometry) |
| fig3fh-dlight-tracks-state-value | blocked, data_missing (photometry) |
| fig3g-dlight-lag | blocked, data_missing (photometry) |
| fig4ac-dopamine-cue-responses | tested |
| fig4bc-dlight-cue-responses | blocked, data_missing (photometry) |
| fig4a-no-dopamine-firing-ramp-before-approach | tested |
| fig4ef-dlight-ramps-before-approach | blocked, data_missing (photometry) |
| fig4e-light-on-alignment | tested |
| fig5a-early-trial-firing-not-reward-rate | tested |
| fig5a-positive-rpe-reward-cue | tested |
| fig5ab-weak-negative-rpe | tested |
| ed8-other-reward-expectation-estimates | tested |
| fig5cd-dlight-positive-not-negative-rpe | blocked, data_missing (photometry) |
| fig5cd-dlight-early-trial-value | blocked, data_missing (photometry) |
| ed7-dlight-470-not-405 | blocked, data_missing (photometry) |
| ed7-non-dopamine-movement-activity | blocked, data_missing (untagged VTA-l units) |

In [ ]:
import resource
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import vta
import vta_measures as vm

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

started = time.time()
units = vta.load_units(port_status=True)
trials = [vta.bandit_trials(u) for u in units]
names = [u["name"] for u in units]
rats = [u["rat"] for u in units]
VTAL = [i for i, u in enumerate(units) if u["rat"] != "IM-1078"]   # VTA-l virus rats
RETRO = [i for i, u in enumerate(units) if u["rat"] == "IM-1078"]  # NAc core virus rat
ALL = list(range(len(units)))
sessions = list(dict.fromkeys(u["session"] for u in units))
first_unit = {s: next(i for i, u in enumerate(units) if u["session"] == s) for s in sessions}
print(f"loaded {len(units)} units of {vta.DATASET} ({vta.VERSION}) in {time.time() - started:.0f} s")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
listing = vta.zip_listing()
md5 = vta.published_md5()
etag = vta.zip_etag()
print("Files in data_and_code.zip:")
for name, size in listing:
    print(f"  {name:40s} {size:>10d} bytes")
print("\nchecksums.md5 gives", md5["data_and_code.zip"], "for data_and_code.zip; the bucket's ETag is", etag,
      "->", "same" if etag == md5["data_and_code.zip"] == vta.ZIP_MD5 else "DIFFERENT")

info = pd.DataFrame({"unit": names, "rat": rats, "session": [u["session"] for u in units],
                     "segments": [" ".join(f"{k}:{len(s['ts'])}" for k, s in u["segments"].items()) for u in units]})
print("\nUnits by rat:")
print(info.groupby("rat").agg(units=("unit", "size"), sessions=("session", "nunique")).to_string())
print("Total units:", len(units), "; rats:", info.rat.nunique(), "; distinct sessions:", len(sessions))
print("Methods: IM657 1, IM1002 3, IM1003 15, IM1037 9 (sum 28, though the text says 27) and IM-1078 2.")
print("Descriptor: 'a total of 29 identified dopaminergic cells recorded from 4 rats'.")
print("\nSegments per unit (spike counts):")
print(info.to_string(index=False))

The bucket serves the published content (ETag equals the MD5 in `checksums.md5`). The data hold
29 units from five rats, not four as the descriptor says: IM-657 1, IM-1002 3, IM-1003 15,
IM-1037 8 and IM-1078 2. The paper's Methods give IM1037 9 units, which with the others sums to 28,
not the 27 the paper states; the data's 27 VTA-l cells match the stated total. Two units lack a
sleep segment and one has an empty one; no analysis here uses sleep.

In [ ]:
rows = []
for s in sessions:
    tr = trials[first_unit[s]]
    same = all(np.array_equal(trials[i]["LightOn"], tr["LightOn"], equal_nan=True)
               and np.array_equal(trials[i]["outcome"], tr["outcome"])
               for i in ALL if units[i]["session"] == s)
    rex = units[first_unit[s]]["segments"]["RL"]["rex"]
    blk_n = pd.Series(tr["block"]).value_counts().sort_index()
    hold = tr["Go"] - tr["CenterIn"]
    clipped = int(np.sum(tr["cni_field_ms"] == 65535))
    diff_ms = np.abs(tr["latency"] * 1000 - tr["cni_field_ms"])
    rt = np.ravel(rex["ReactionTime"]) / 1000
    rows.append({"session": s, "units": sum(units[i]["session"] == s for i in ALL), "same_events_all_units": same,
                 "attempts": tr["n_attempts"], "trials": len(tr["LightOn"]), "rewarded": round(tr["outcome"].mean(), 2),
                 "blocks": len(blk_n), "trials_per_block_but_last": f"{blk_n.iloc[:-1].min()}-{blk_n.iloc[:-1].max()}",
                 "probs": " ".join(sorted({f"{p:g}" for p in np.ravel(rex["block_probabilities"])})),
                 "hold_s": f"{np.nanmin(hold):.2f}-{np.nanmax(hold):.2f}",
                 "latency_s": f"{np.nanmin(tr['latency']):.2f}/{np.nanmedian(tr['latency']):.2f}/{np.nanmax(tr['latency']):.0f}",
                 "CNI_clipped": clipped, "CNI_mismatch_gt5ms": int(np.sum(diff_ms > 5)) - clipped,
                 "RT_vs_events_max_ms": round(float(np.nanmax(np.abs(rt - (tr["CenterOut"] - tr["Go"])))) * 1000, 1)})
sess = pd.DataFrame(rows)
print(sess.to_string(index=False))
print("Completed trials whose latency is missing or not positive:",
      {s: int(np.sum(~(trials[first_unit[s]]["latency"] > 0))) for s in sessions if np.any(~(trials[first_unit[s]]["latency"] > 0))})
print("\nTrials in all sessions:", sess.trials.sum(), "; reward probabilities used:", sorted(set(" ".join(sess.probs).split())))

Events and outcomes are identical for units of the same session, so each session's behaviour is
counted once: 21 sessions, 6190 completed trials. Every session used reward probabilities of 10, 50
and 90% (Methods: 10, 50, 90% for electrophysiology rats). Blocks hold 36 to 46 completed trials
(the last block of each session, cut short, left out), against the Methods' 35 to 45: consistent up
to one trial. The hold from Center-In to the Go cue is 0.5 to 1.5 s, as the Methods say. The
descriptor's `ReactionTime` matches Center-Out minus Go cue within 0.5 ms in 18 sessions and differs
by up to 273 ms in 3. The descriptor's `CNIlatency` ("latency to center-nose-in") equals Center-In
minus Light-On except in the columns counted: values above 65.535 s are stored as 65535 ms, the
largest 16-bit integer, and in 4 sessions one other value differs (in IM-657_160613 the last trial
carries the previous trial's value). Latency is therefore computed here from the event times; one
trial has none and is left out of the fits.

In [ ]:
rows = []
for i, u in enumerate(units):
    seg = u["segments"]["RL"]
    raw_unsorted = {k: (not s["ts_sorted"]) for k, s in u["segments"].items()}
    ts = seg["ts"]
    tr = trials[i]
    t0, t1 = np.nanmin(tr["LightOn"]), np.nanmax(tr["SideOut"])
    rate_bandit = vta.counts_in(ts, np.array([t0]), np.array([t1]))[0] / (t1 - t0)
    # segment durations are not deposited (the Sleep segment has no events), so each segment's last
    # spike time stands in for its length, which slightly overstates the rate
    spans = {k: max(s["ts"][-1] if len(s["ts"]) else 0, 0) for k, s in u["segments"].items()}
    n_all = sum(len(s["ts"]) for s in u["segments"].values())
    rows.append({"unit": u["name"], "FR_field": round(u["FR"], 2), "rate_bandit": round(rate_bandit, 2),
                 "rate_all_segments": round(n_all / sum(spans.values()), 2),
                 "rate_RL_0_to_last_spike": round(len(ts) / ts[-1], 2),
                 "unsorted_segments": ",".join(k for k, v in raw_unsorted.items() if v) or "-",
                 "isi_lt_1ms_RL": int(np.sum(np.diff(ts) < 0.001))})
rates = pd.DataFrame(rows)
print(rates.to_string(index=False))
inv = pd.DataFrame([{"unit": u["name"], "segment": k, "n_inversions": s["n_inversions"], "max_inversion_ms": round(1000 * s["max_inversion"], 3),
                     "first_inversion_s": s["first_inversion"]} for u in units for k, s in u["segments"].items() if not s["ts_sorted"]])
print(f"\nUnits with spike times out of order in one or more segments: {inv.unit.nunique()} of {len(units)}")
print(inv.to_string(index=False))
print("\nFR field against the bandit rate from first Light-On to last Side-Out: max |diff|",
      round((rates.FR_field - rates.rate_bandit).abs().max(), 2), "Hz; against spikes / last spike time of RL:",
      round((rates.FR_field - rates.rate_RL_0_to_last_spike).abs().max(), 2), "Hz")

# recordings deposited twice: coincident spikes between units of the same session
dup = []
for s in sessions:
    ids = [i for i in ALL if units[i]["session"] == s]
    for a in range(len(ids)):
        for b in range(a + 1, len(ids)):
            A, B = units[ids[a]]["segments"]["RL"]["ts"], units[ids[b]]["segments"]["RL"]["ts"]
            k = np.clip(np.searchsorted(B, A), 1, len(B) - 1)
            near = np.minimum(np.abs(A - B[k]), np.abs(A - B[k - 1])) < 0.0005
            dup.append({"session": s, "a": names[ids[a]], "b": names[ids[b]], "frac_a_within_0.5ms_of_b": round(near.mean(), 4)})
print("\nUnit pairs recorded in the same session:")
print(pd.DataFrame(dup).to_string(index=False))

# the same tetrode on different days: waveform similarity
def tet(n):
    return n[:-1].split("t")[-1]
wf = []
for a in ALL:
    for b in ALL:
        if b > a and rats[a] == rats[b] and tet(names[a]) == tet(names[b]) and units[a]["session"] != units[b]["session"]:
            wf.append({"a": names[a], "b": names[b], "waveform_r": round(np.corrcoef(np.ravel(units[a]["meanWave"]), np.ravel(units[b]["meanWave"]))[0, 1], 3),
                       "rate_a": round(units[a]["FR"], 2), "rate_b": round(units[b]["FR"], 2)})
wf = pd.DataFrame(wf)
print("\nUnits on the same tetrode of the same rat on different days:", len(wf))
print(wf.sort_values("waveform_r", ascending=False).head(12).to_string(index=False))

The descriptor's `FR` ("session-wide average firing rate") is close to, but not the same as, the
bandit rate computed between the first Light-On and the last Side-Out, and it equals none of the
rates computed here exactly; its exact definition is not given. The rate over all segments divides by each segment's last spike
time, as the segments' durations are not deposited, so it slightly overstates the rate. In 11 units the spike times in one
or more segments are not in ascending order (inversions of at most 0.7 ms, for example 2399.4923 s
after 2399.4926 s in IM657160613t11a); they are sorted on loading.
No pair of units from the same session shares more than 1.4% of its spikes within 0.5 ms, so no
recording is deposited twice. Units from the same tetrode on different days (optrodes were lowered
at least 80 um after each session, Methods) are treated as distinct cells, as the paper does; their
mean waveforms are shown above, and this is stated as an assumption where cells are counted.

In [ ]:
rows = []
for i, u in enumerate(units):
    p = u["segments"]["Pav"]
    rex, box = p["rex"], p["boxts"]
    ev = np.asarray(box["event"])
    tone = np.ravel(rex["Tone"])
    cc = {int(f): int(q) for f, q in np.asarray(rex["Cue_Contingency"])}
    food = np.ravel(rex["Food"])
    rows.append({"unit": names[i], "contingency": " ".join(f"{f}Hz:{q}%" for f, q in cc.items() if f) + f" none:{cc[0]}%",
                 "trials": len(tone), "click_minus_onset_s": round(float(np.nanmedian(ev[1] - ev[0])), 3),
                 "onset_interval_s": f"{np.nanmin(np.diff(ev[0])):.1f}-{np.nanmax(np.diff(ev[0])):.1f}",
                 "port_window": str(rex["port_win"]), "port_samples": rex["port"].shape[1]})
pav_info = pd.DataFrame(rows)
print(pav_info.to_string(index=False))
grp = pav_info.loc[VTAL, "contingency"].str.contains("75%").map({True: "75/25/0", False: "100/50/0"})
print("\nVTA-l cells by Pavlovian tone set:", grp.value_counts().to_dict(),
      "; Extended Data Fig. 5b: 75%/25% n=9 cells, 100%/50% n=18")
# port occupancy around the click on unpredicted trials (is the port window aligned to trial onset?)
u = units[VTAL[-1]]
p = u["segments"]["Pav"]
port = p["rex"]["port"].astype(float)
tone = np.ravel(p["rex"]["Tone"])
tgrid = np.arange(port.shape[1]) / 1000 + p["rex"]["port_win"][0]
for f in np.unique(tone):
    m = port[tone == f].mean(0)
    print(f"  {names[VTAL[-1]]} tone {int(f)} Hz: port occupancy 0-2 s {m[(tgrid >= 0) & (tgrid < 2)].mean():.2f}, "
          f"3.2-5 s {m[(tgrid >= 3.2) & (tgrid < 5)].mean():.2f}")

rows = []
for i, u in enumerate(units):
    L = u["segments"]["Laser"]
    rex = L["rex"]
    tt = np.ravel(rex["trialType"]).astype(int)
    on = np.ravel(L["boxts"]["LaserOn"])
    pw, fq = np.ravel(rex["pulseWidth"]), np.ravel(rex["freq"])
    rows.append({"unit": names[i], "conditions (ms@Hz)": " ".join(f"{pw[k - 1]:g}@{fq[k - 1]:g}" for k in np.unique(tt)),
                 "pulses": len(on)})
print("\nLaser conditions per unit (pulse width in ms at frequency in Hz, as `pulseWidth` and `freq`):")
print(pd.DataFrame(rows).to_string(index=False))

The Pavlovian event table has two rows: trial onset (the tone onset; for unpredicted rewards a
nominal onset with no tone) and the food delivery click, 3.136 s later on rewarded trials, which
fits 2.6 s of tone plus a 0.5 s delay (Methods). Trial onsets are 18 to 33 s apart, which fits the
Methods' 15 to 30 s intertrial interval added to the 3.1 s of cue and delay. The food-port status is sampled at 30 kHz from 2 s before to 10 s after
trial onset (here kept at 1 kHz); occupancy rises after the click on unpredicted trials, which is
consistent with alignment on trial onset. Among the 27 VTA-l cells, 19 had 100/50/0% tones and 8
had 75/25/0%, against the 18 and 9 of the Extended Data Fig. 5 legend. The laser pulse widths
(stated in ms by the descriptor) include 5 and 10 ms pulses at 2 Hz for 27 units; units
IM657160613t11a and IM1002170114t05b were given pulses of at most 1 ms.

In [ ]:
# Shared clock: contrasts that a response shows and an offset does not.
rng_clock = np.random.default_rng(11)
rows = []
for i, u in enumerate(units):
    ts = u["segments"]["RL"]["ts"]
    tr = trials[i]
    rew = tr["outcome"] == 1
    def contrast(shift):
        si = tr["SideIn"] + shift
        r = vta.counts_in(ts, si + 0.05, si + 0.25) / 0.2
        return stats.mannwhitneyu(r[rew], r[~rew], alternative="greater").pvalue, r[rew].mean() - r[~rew].mean()
    p0, d0 = contrast(0.0)
    p5, d5 = contrast(-5.0)
    lab = rng_clock.permutation(rew)
    si = tr["SideIn"]
    r = vta.counts_in(ts, si + 0.05, si + 0.25) / 0.2
    pperm = stats.mannwhitneyu(r[lab], r[~lab], alternative="greater").pvalue
    L = u["segments"]["Laser"]
    on = np.ravel(L["boxts"]["LaserOn"])
    lts = L["ts"]
    f_on = np.mean(vta.counts_in(lts, on, on + 0.01) > 0)
    f_pre = np.mean(vta.counts_in(lts, on - 0.2, on - 0.19) > 0)
    rows.append({"unit": names[i], "rew-unrew Hz": round(d0, 1), "p": p0, "shift -5 s Hz": round(d5, 1), "p_shift": p5,
                 "p_labels_shuffled": pperm, "laser: P(spike 0-10 ms)": round(f_on, 2), "P(spike -200..-190 ms)": round(f_pre, 2)})
clock = pd.DataFrame(rows)
print(clock.to_string(index=False, float_format=lambda x: f"{x:.2g}"))
print("\nUnits with rewarded > unrewarded firing 50-250 ms after Side-In at p < 0.01:", int((clock.p < 0.01).sum()),
      "of 29; with Side-In shifted 5 s earlier:", int((clock.p_shift < 0.01).sum()),
      "; with outcome labels shuffled:", int((clock.p_labels_shuffled < 0.01).sum()))

The spikes share the clock of the bandit events: in 26 of 29 units firing 50 to 250 ms after
Side-In is higher on rewarded than on unrewarded trials (p < 0.01), against 1 of 29 with the events
shifted 5 s earlier and 1 of 29 with the outcome labels shuffled. In the three other units
(IM1002170118t06a, IM1003170307t01a, IM1003170309t01a) this contrast does not show the clock; their
Go cue responses in the fig4ac section do. In every unit laser pulses are followed by a spike within
10 ms more often than the same 10 ms window 200 ms earlier, so the laser segment's spikes and pulses
share a clock too. The Pavlovian clock is checked by the click responses in the
Extended Data Fig. 5 section below.

## Reward rate: the leaky integrator's time constant per session

Reward rate (Methods, Behavior) is a leaky integrator incremented at each reward and decaying with
time constant tau, chosen per session from 1 to 1200 s to make its correlation with log(latency)
most negative. It is used by the claims of Figures 1, 2 and 5; `vta.py` says where each detail
comes from.

In [ ]:
TAU = {}
for s in sessions:
    TAU[s] = vta.best_tau(trials[first_unit[s]])
tau_unit = [TAU[u["session"]][0] for u in units]
print(pd.DataFrame({"session": sessions, "tau_s": [TAU[s][0] for s in sessions],
                    "r_at_tau": [round(TAU[s][1], 3) for s in sessions]}).to_string(index=False))

# reward rate of each trial at Light-On and at Side-In (before that trial's outcome)
rr_lighton = [vta.leaky_rate(trials[i]["LightOn"], vta.reward_times(trials[i]), [tau_unit[i]])[0] for i in ALL]
rr_sidein = [vta.leaky_rate(trials[i]["SideIn"], vta.reward_times(trials[i]), [tau_unit[i]])[0] for i in ALL]

## fig1b-latency-scales-with-reward-rate

> Rats were more motivated when rewards were more likely; their latency from Light-On to
> Center-In scaled inversely with reward rate.

Criterion: per session, Pearson's r of reward rate (best tau) with log(latency); reproduced when r
is negative in more than half of the sessions and its mean is negative by a one-sample t-test,
p < 0.05. Since tau is chosen to make r negative, the notebook also gives the null: the same
selection on latencies shuffled within the session (200 shuffles per session, on a grid of 76
log-spaced taus from 1 to 1200 s, a coarser grid to keep the run short).

In [ ]:
rng = np.random.default_rng(101)
coarse = np.unique(np.round(np.geomspace(1, 1200, 80)))
rows = []
for s in sessions:
    tr = trials[first_unit[s]]
    tau, r, rs = TAU[s]
    rr = vta.leaky_rate(tr["LightOn"], vta.reward_times(tr), [tau])[0]
    y = vta.log_latency(tr)
    ok = np.isfinite(y)
    p = stats.pearsonr(rr[ok], y[ok]).pvalue
    null = []
    for _ in range(200):
        tr2 = dict(tr)
        tr2["latency"] = rng.permutation(tr["latency"])
        null.append(vta.best_tau(tr2, coarse)[1])
    null = np.array(null)
    rows.append({"session": s, "rat": s.split("_")[0], "trials": int(ok.sum()), "latency_0_dropped": int((~ok).sum()), "tau_s": tau, "r": r, "p": p,
                 "null_r_median": np.median(null), "null_r_1pct": np.quantile(null, 0.01),
                 "r_below_null_1pct": r < np.quantile(null, 0.01)})
f1 = pd.DataFrame(rows)
print(f1.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
t1 = stats.ttest_1samp(f1.r, 0)
print(f"\nSessions: {len(f1)}; r < 0 in {int((f1.r < 0).sum())}; mean r {f1.r.mean():.3f} "
      f"(one-sample t = {t1.statistic:.2f}, p = {t1.pvalue:.2g}); median best tau {f1.tau_s.median():.0f} s "
      f"(range {f1.tau_s.min():.0f}-{f1.tau_s.max():.0f}); at the grid's edge: {int(f1.tau_s.isin([1, 1200]).sum())}")
print(f"Sessions whose r is below the 1st percentile of the shuffled-latency null: {int(f1.r_below_null_1pct.sum())} of {len(f1)}")
print("By rat:")
print(f1.groupby("rat").agg(sessions=("r", "size"), mean_r=("r", "mean"), negative=("r", lambda x: int((x < 0).sum()))).round(3).to_string())
fixed = f1.tau_s.median()
rfix = []
for s in sessions:
    tr = trials[first_unit[s]]
    y = vta.log_latency(tr)
    ok = np.isfinite(y)
    rfix.append(stats.pearsonr(vta.leaky_rate(tr["LightOn"], vta.reward_times(tr), [fixed])[0][ok], y[ok])[0])
print(f"Variant, one tau for all sessions ({fixed:.0f} s): r < 0 in {int(np.sum(np.array(rfix) < 0))} of {len(rfix)}, mean r {np.mean(rfix):.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
s = sessions[3]
tr = trials[first_unit[s]]
rr = vta.leaky_rate(tr["LightOn"], vta.reward_times(tr), [TAU[s][0]])[0]
ax[0].scatter(rr, tr["latency"], s=6)
ax[0].set_yscale("log")
ax[0].set_xlabel(f"reward rate (rewards, tau {TAU[s][0]:.0f} s)")
ax[0].set_ylabel("latency (s)")
ax[0].set_title(s)
ax[1].hist(f1.r, bins=12)
ax[1].axvline(0, color="k")
ax[1].set_xlabel("r(reward rate, log latency) per session")
plt.tight_layout()
plt.show()
print(f"Left: latency range {tr['latency'].min():.2f}-{tr['latency'].max():.1f} s; right: r range {f1.r.min():.3f} to {f1.r.max():.3f}")

**Result: reproduced.** In all 21 sessions the correlation of reward rate with log(latency) is
negative (mean r -0.307, one-sample t = -12.50, p = 6.6e-11; range -0.496 to -0.136), with best tau
from 12 to 295 s and none at the edge of the 1 to 1200 s grid. In 19 of 21 sessions r lies below
the 1st percentile of the null obtained by repeating the selection of tau on shuffled latencies,
so the correlation is not a product of choosing tau; with a single tau of 58 s for every session,
r is negative in 21 of 21 (mean -0.291).

## fig2-tagged-dopamine-cells

> Optogenetic tagging identified 27 well-isolated VTA-l cells with reliable short-latency
> light-evoked spikes as dopamine neurons, plus 2 retrogradely tagged cells from a rat with NAc
> core virus.

Criterion: SALT p < 0.001 for the 5 ms and 10 ms pulse trains, and Pearson r > 0.9 between the
light-evoked and session-wide waveforms. Only the 29 deposited cells are in the data set, so the
test is whether each of them passes, not how many of all recorded units would. For the two units
that received no 5 or 10 ms pulses, the 2 Hz trains with their longest pulses (1 ms) are tested in
their place. Which wires the waveform correlation uses is not given; readings are the four wires
concatenated and the wire with the largest amplitude. A chance reference is the same SALT test
with the pulse times moved 250 ms later, to the middle of the interval between pulses.

In [ ]:
rows = []
for i, u in enumerate(units):
    L = u["segments"]["Laser"]
    rex = L["rex"]
    tt = np.ravel(rex["trialType"]).astype(int)
    on = np.ravel(L["boxts"]["LaserOn"])
    pw, fq = np.ravel(rex["pulseWidth"]), np.ravel(rex["freq"])
    conds = [k for k in np.unique(tt) if fq[k - 1] == 2 and pw[k - 1] in (5, 10)]
    substituted = not conds
    if substituted:
        w = max(pw[k - 1] for k in np.unique(tt) if fq[k - 1] == 2)
        conds = [k for k in np.unique(tt) if fq[k - 1] == 2 and pw[k - 1] == w]
    ps = {f"{pw[k - 1]:g}ms#{k}": vta.salt(L["ts"], on[tt == k])[0] for k in conds}
    p_sham = max(vta.salt(L["ts"], on[tt == k] + 0.25)[0] for k in conds)
    mw, le = np.asarray(u["meanWave"]), np.asarray(u["meanLightEvoked"])
    big = int(np.argmax(mw.max(0) - mw.min(0)))
    rows.append({"unit": names[i], "rat": rats[i], "pulses_tested": " ".join(ps), "substituted_1ms": substituted,
                 "salt_p_max": max(ps.values()), "sham_p_min": p_sham,
                 "r_wave_4wires": np.corrcoef(mw.ravel(), le.ravel())[0, 1],
                 "r_wave_largest": np.corrcoef(mw[:, big], le[:, big])[0, 1]})
tag = pd.DataFrame(rows)
tag["light_responsive"] = tag.salt_p_max < 0.001
tag["pass_4wires"] = tag.light_responsive & (tag.r_wave_4wires > 0.9)
tag["pass_largest"] = tag.light_responsive & (tag.r_wave_largest > 0.9)
print(tag.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
for col in ("light_responsive", "pass_4wires", "pass_largest"):
    print(f"{col}: VTA-l {int(tag.loc[VTAL, col].sum())} of {len(VTAL)}, IM-1078 {int(tag.loc[RETRO, col].sum())} of {len(RETRO)}")
strict = ~tag.substituted_1ms
print("Variant, the two units with no 5 or 10 ms pulses counted as failing (no 1 ms substitution):",
      {col: (int((tag[col] & strict).loc[VTAL].sum()), int((tag[col] & strict).loc[RETRO].sum())) for col in ("light_responsive", "pass_largest", "pass_4wires")})
print("Sham pulses (250 ms later) with SALT p < 0.001:", int((tag.sham_p_min < 0.001).sum()), "of 29")
for i in ALL:
    if not tag.light_responsive[i]:
        L = units[i]["segments"]["Laser"]
        tt = np.ravel(L["rex"]["trialType"]).astype(int)
        on = np.ravel(L["boxts"]["LaserOn"])
        pw, fq = np.ravel(L["rex"]["pulseWidth"]), np.ravel(L["rex"]["freq"])
        print(f"{names[i]}: SALT p for every 2 Hz condition:",
              {f"{pw[k - 1]:g}ms#{k}": round(vta.salt(L["ts"], on[tt == k])[0], 4) for k in np.unique(tt) if fq[k - 1] == 2})
print("Per rat (light responsive):", tag.groupby("rat").light_responsive.agg(["sum", "size"]).to_dict("index"))

**Result: partially reproduced.** SALT gives p < 0.001 for 26 of the 27 VTA-l cells and for both
IM-1078 cells, and for none of the 29 with sham pulse times. IM1003170321t01c fails with its 5 and
10 ms trains (p = 0.087 and 0.05), though two of its 1 ms trains give p < 0.001. Two units received
no 5 or 10 ms pulses and were tested on their 1 ms trains, a substitution the criterion does not
make; counting them as failing gives 24 light-responsive VTA-l cells. The waveform criterion (r > 0.9,
which the Methods say all tagged cells met) holds for 23 of the 27 on the largest wire and 18 on
four wires, so the cells passing both tests are 23 and 2 (largest wire) or 18 and 1 (four wires),
and 22 and 2 or 17 and 1 without the 1 ms substitution. The retrograde count (2) matches; no VTA-l
total is within 27 +- 2.7. The data set holds only the cells the paper selected, so the selection
from all recorded units cannot be repeated.

## fig2d-dopamine-firing-rates

> Identified dopamine neurons were tonically active with low firing rates, mean 7.7 Hz, range 3.7
> to 12.9 Hz.

Criterion: session-wide firing rate per cell; the set (27 or 29) and the segment (bandit only or
the whole recording) are open and each is run. The descriptor's `FR` field is shown too.

In [ ]:
rows = []
for label, ids in (("27 VTA-l", VTAL), ("all 29", ALL)):
    for col in ("rate_bandit", "rate_all_segments", "FR_field"):
        v = rates.loc[ids, col]
        rows.append({"cells": label, "rate": col, "mean": v.mean(), "min": v.min(), "max": v.max(),
                     "mean_ok": abs(v.mean() - 7.7) <= 0.77, "min_ok": abs(v.min() - 3.7) <= 0.37, "max_ok": abs(v.max() - 12.9) <= 1.29})
fr = pd.DataFrame(rows)
print(fr.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
print("\nBy rat (bandit rate):")
print(rates.assign(rat=rats).groupby("rat").rate_bandit.agg(["size", "mean", "min", "max"]).round(2).to_string())
cv = [np.std(np.diff(units[i]["segments"]["RL"]["ts"])) / np.mean(np.diff(units[i]["segments"]["RL"]["ts"])) for i in ALL]
print(f"\nISI coefficient of variation in the bandit task: median {np.median(cv):.2f}, range {min(cv):.2f}-{max(cv):.2f}")

**Result: reproduced.** Over the bandit task the 27 VTA-l cells fire at a mean of 7.77 Hz, range
3.77 to 12.93 Hz (paper: 7.7, 3.7 to 12.9); all 29 give 7.72 Hz with the same range. Over all
segments the mean and minimum match (7.41, 3.59 Hz) but the maximum, 11.60 Hz, is outside 12.9 +-
1.29; the descriptor's `FR` field gives 7.36, 3.62 and 11.73 Hz, within tolerance.

## Per-unit measures

One pass over the 29 units computes the per-trial measures and mean traces of the claims below
(`vta_measures.py`), including the time-to-cue-response shuffle tests, with a generator per unit.

In [ ]:
t_pass = time.time()
M = [vm.bandit_pass(units[i], trials[i], np.random.default_rng([2019, i])) for i in ALL]
P = [vm.pav_pass(units[i]) for i in ALL]
print(f"per-unit pass: {time.time() - t_pass:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024:.0f} MB")

## ed5-pavlovian-dopamine-click-responses

> In the Pavlovian task, dopamine cells responded strongly to randomly timed food hopper clicks
> and progressively less strongly when the clicks were made more predictable by preceding cues.

Criterion: peak firing (spike density) in the 0.5 s after the click per rewarded trial, by
condition (unpredicted, medium, high); one-way ANOVA per cell. Reproduced when the mean over the 27
cells is ordered unpredicted > medium > high and the ANOVA has p < 0.05 in more than half of the
cells. Expected cell counts: 9 with 75/25% tones and 18 with 100/50%. A 50 ms binned histogram is
the variant.

In [ ]:
def click_table(ids, binned=False, key="click_peak"):
    rows = []
    for i in ids:
        p = P[i]
        ts = units[i]["segments"]["Pav"]["ts"]
        vals = {}
        for c in ("unpredicted", "medium", "high"):
            sel = (p["cond"] == c) & (p["food"] == 1) & np.isfinite(p["click"])
            if binned:
                e = p["click"][sel]
                vals[c] = np.max(np.stack([vta.counts_in(ts, e + a, e + a + 0.05) / 0.05 for a in np.arange(0, 0.5, 0.05)], 1), 1)
            else:
                vals[c] = p[key][sel]
        f = stats.f_oneway(*vals.values())
        rows.append({"unit": names[i], "tones": f"{p['probs']['high']:g}/{p['probs']['medium']:g}",
                     **{c: v.mean() for c, v in vals.items()}, **{f"n_{c}": len(v) for c, v in vals.items()},
                     "F": f.statistic, "p": f.pvalue})
    return pd.DataFrame(rows)
ck = click_table(VTAL)
print(ck.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
m = ck[["unpredicted", "medium", "high"]].mean()
print("\nMean peak over the 27 cells (Hz):", m.round(1).to_dict(), "; ordered unpredicted > medium > high:",
      bool(m.unpredicted > m.medium > m.high))
print("Cells with ANOVA p < 0.05:", int((ck.p < 0.05).sum()), "of", len(ck),
      "; cells individually ordered unpredicted > medium > high:", int(((ck.unpredicted > ck.medium) & (ck.medium > ck.high)).sum()))
print("By tone set:", ck.groupby("tones")[["unpredicted", "medium", "high"]].mean().round(1).to_dict("index"),
      "; cells per tone set:", ck.tones.value_counts().to_dict())
ckb = click_table(VTAL, binned=True)
mb = ckb[["unpredicted", "medium", "high"]].mean()
print("Variant, 50 ms bins: means", mb.round(1).to_dict(), "; p < 0.05 in", int((ckb.p < 0.05).sum()), "cells")
ck4 = click_table(VTAL, key="click_peak_sd4")
m4 = ck4[["unpredicted", "medium", "high"]].mean()
print("Variant, SD 4.5 ms kernel (the claims file's primary reading of 'variance 20ms'): means", m4.round(1).to_dict(),
      "; ordered:", bool(m4.unpredicted > m4.medium > m4.high), "; p < 0.05 in", int((ck4.p < 0.05).sum()), "of 27")
ck29 = click_table(ALL)
print("Variant, 29 cells: means", ck29[["unpredicted", "medium", "high"]].mean().round(1).to_dict(), "; p < 0.05 in", int((ck29.p < 0.05).sum()))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
t = np.arange(-1.0, 1.5 + 1e-9, 0.001)
for c in ("unpredicted", "medium", "high"):
    ax[0].plot(t, np.nanmean([P[i]["trace_click"][c] for i in VTAL], 0), label=c)
ax[0].set_xlabel("time from click (s)")
ax[0].set_ylabel("Hz")
ax[0].legend()
t = np.arange(-1.0, 4.0 + 1e-9, 0.001)
for c in ("zero", "medium", "high"):
    ax[1].plot(t, np.nanmean([P[i]["trace_cue"][c] for i in VTAL], 0), label=c)
ax[1].set_xlabel("time from cue onset (s)")
ax[1].legend()
plt.tight_layout()
plt.show()
print("Mean traces of the 27 cells; ranges (Hz): click", round(np.nanmin([np.nanmin(P[i]['trace_click'][c]) for i in VTAL for c in ('unpredicted', 'medium', 'high')]), 1),
      "to", round(np.nanmax([np.nanmean([P[i]["trace_click"][c] for i in VTAL], 0).max() for c in ("unpredicted", "medium", "high")]), 1))

**Result: partially reproduced.** The mean click response over the 27 cells is ordered as the
claim states: unpredicted 47.3, medium 42.3, high 37.5 Hz (the same order with 50 ms bins, with the
SD 4.5 ms kernel and with 29 cells), and 16 cells show that order individually. The per-cell ANOVA
gives p < 0.05 in 12 of 27 cells (10 with 50 ms bins, 7 with the SD 4.5 ms kernel), not more than
half, so that part of the criterion fails under every kernel. The data hold 8 cells with 75/25%
tones and 19 with 100/50%; 19 is within 18 +- 1.8 and 8 is outside 9 +- 0.9. The miss of 8 against
9 follows from the data holding 8 IM-1037 cells where the Methods give 9 (data check).

## ed5-pavlovian-dopamine-cue-responses

> Dopamine cell responses to the Pavlovian cues followed the cues' reward probability, in line with
> RPE-like coding.

Criterion: peak firing in the 0.5 s after cue onset per trial, by tone; one-way ANOVA per cell.
Reproduced when the mean over the 27 cells is ordered high > medium > zero and the ANOVA has
p < 0.05 in more than half of the cells.

In [ ]:
def cue_table(key="cue_peak"):
    rows = []
    for i in ALL:
        p = P[i]
        vals = {c: p[key][(p["cond"] == c) & np.isfinite(p[key])] for c in ("zero", "medium", "high")}
        f = stats.f_oneway(*vals.values())
        rows.append({"unit": names[i], "tones": f"{p['probs']['high']:g}/{p['probs']['medium']:g}",
                     **{c: v.mean() for c, v in vals.items()}, "F": f.statistic, "p": f.pvalue})
    return pd.DataFrame(rows)
cue = cue_table()

cue27 = cue.loc[VTAL]
print(cue27.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
m = cue27[["zero", "medium", "high"]].mean()
print("\nMean peak over the 27 cells (Hz):", m.round(1).to_dict(), "; ordered high > medium > zero:", bool(m.high > m.medium > m.zero))
print("Cells with ANOVA p < 0.05:", int((cue27.p < 0.05).sum()), "of 27; cells ordered high > medium > zero:",
      int(((cue27.high > cue27.medium) & (cue27.medium > cue27.zero)).sum()))
print("By tone set:", cue27.groupby("tones")[["zero", "medium", "high"]].mean().round(1).to_dict("index"))
cue4 = cue_table("cue_peak_sd4").loc[VTAL]
m4 = cue4[["zero", "medium", "high"]].mean()
print("Variant, SD 4.5 ms kernel: means", m4.round(1).to_dict(), "; ordered:", bool(m4.high > m4.medium > m4.zero),
      "; p < 0.05 in", int((cue4.p < 0.05).sum()), "of 27")
print("Variant, 29 cells: means", cue[["zero", "medium", "high"]].mean().round(1).to_dict(), "; p < 0.05 in", int((cue.p < 0.05).sum()))

**Result: partially reproduced.** The mean cue response over the 27 cells follows the reward
probability: zero 29.6, medium 31.8, high 36.3 Hz, the same order in both tone sets, with the SD
4.5 ms kernel, and in 16 cells individually. The per-cell ANOVA gives p < 0.05 in 10 of 27 cells
(4 with the SD 4.5 ms kernel), not more than half.

## ed5a-pavlovian-behavior

> Rats learned the cue probabilities; in an example session the rat entered the food port more
> during the 75% cue than the 25% and 0% cues.

Criterion: food-port head entry 0.3 to 3 s after cue onset per trial, one-way ANOVA across tones,
in every Pavlovian session with 75/25/0% tones (each session once). Readings for the per-trial
measure: the fraction of the epoch at the port, and whether the rat was at the port at any time in
it. Reproduced when the 75% cue has the highest head-entry rate and p < 0.05 in most of those
sessions, and at least one session gives p < 1.2e-6. Sessions with 100/50/0% tones are shown as a
variant.

In [ ]:
rows = []
for s in sessions:
    i = first_unit[s]
    p = units[i]["segments"]["Pav"]
    port = p["rex"]["port"].astype(float)
    tgrid = np.arange(port.shape[1]) / 1000 + p["rex"]["port_win"][0]
    w = (tgrid >= 0.3) & (tgrid < 3.0)
    frac, anyt = port[:, w].mean(1), port[:, w].max(1)
    cond = P[i]["cond"]
    for label, x in (("fraction", frac), ("any", anyt)):
        groups = {c: x[cond == c] for c in ("high", "medium", "zero")}
        f = stats.f_oneway(*groups.values())
        rows.append({"session": s, "tones": f"{P[i]['probs']['high']:g}/{P[i]['probs']['medium']:g}/0", "measure": label,
                     **{c: g.mean() for c, g in groups.items()}, "F": f.statistic, "p": f.pvalue,
                     "high_highest": groups["high"].mean() > max(groups["medium"].mean(), groups["zero"].mean())})
beh = pd.DataFrame(rows)
print(beh.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
for label in ("fraction", "any"):
    b = beh[(beh.measure == label) & (beh.tones == "75/25/0")]
    print(f"\n75/25/0 sessions, {label}: {len(b)} sessions; 75% highest in {int(b.high_highest.sum())}; p < 0.05 in {int((b.p < 0.05).sum())}; "
          f"p < 1.2e-6 in {int((b.p < 1.2e-6).sum())}; largest F {b.F.max():.1f}, smallest p {b.p.min():.2g}")
    b = beh[(beh.measure == label) & (beh.tones != "75/25/0")]
    print(f"variant, 100/50/0 sessions, {label}: {len(b)} sessions; high cue highest in {int(b.high_highest.sum())}; p < 0.05 in {int((b.p < 0.05).sum())}")

**Result: reproduced.** In 5 of the 7 sessions with 75/25/0% tones the rat is at the food port
most during the 75% cue and the ANOVA gives p < 0.05, and in all 5 p < 1.2e-6 (smallest p 1.1e-17,
largest F 55.6, for the fraction of the epoch at the port; the same counts for any entry in the
epoch). In the 14 sessions with 100/50/0% tones the high cue leads in 11 but the ANOVA reaches
p < 0.05 in only 2 (fraction) or 5 (any entry).

## fig2efgh-tonic-firing-indifferent-to-reward-rate

> Tonic dopamine cell firing in each block was indifferent to reward rate, with no significant
> change in the firing rate of any individual dopamine cell between higher- and lower-reward blocks.

Criterion: a claim of absence. Spikes in 1 min bins; blocks split per session into high and low by
the median of the block-averaged reward rate. Readings for the per-cell test: (a) rank-sum of the
1 min bin rates in high against low blocks, (b) rank-sum of the block-mean rates; p < 0.05.
Reproduced under a reading when no cell is significant. Also shown: the count against chance
(binomial), each cell's difference in Hz, and a population signed-rank test. The 1 min bins are
laid from the session's first Light-On; a bin counts for a block when it lies wholly inside it. With
an odd number of blocks the median block is dropped (`median_block="low"` is the variant). Because
the 1 min bins of a session are not independent, reading (a) is also run against circular shifts of
the bins' high/low labels, and for reading (b) the smallest p its numbers of blocks allow is shown.

In [ ]:
def tonic(i, median_block="drop"):
    ts = units[i]["segments"]["RL"]["ts"]
    tr = trials[i]
    high, low, means = vta.block_split(tr, tau_unit[i], median_block)
    iv = vta.block_intervals(tr)
    t0 = np.nanmin(tr["LightOn"])
    edges = np.arange(t0, max(e for _, e in iv.values()), 60.0)
    bs, be = edges[:-1], edges[1:]
    rate = vta.counts_in(ts, bs, be) / 60.0
    def bins_of(blocks):
        keep = np.zeros(len(bs), bool)
        for b in blocks:
            a, e = iv[b]
            keep |= (bs >= a) & (be <= e)
        return rate[keep]
    bh, bl = bins_of(high), bins_of(low)
    mh = np.array([vta.counts_in(ts, np.array([iv[b][0]]), np.array([iv[b][1]]))[0] / (iv[b][1] - iv[b][0]) for b in high])
    ml = np.array([vta.counts_in(ts, np.array([iv[b][0]]), np.array([iv[b][1]]))[0] / (iv[b][1] - iv[b][0]) for b in low])
    dur_h = sum(iv[b][1] - iv[b][0] for b in high)
    dur_l = sum(iv[b][1] - iv[b][0] for b in low)
    rate_h = sum(vta.counts_in(ts, np.array([iv[b][0]]), np.array([iv[b][1]]))[0] for b in high) / dur_h
    rate_l = sum(vta.counts_in(ts, np.array([iv[b][0]]), np.array([iv[b][1]]))[0] for b in low) / dur_l
    keep_h = np.zeros(len(bs), bool)
    keep_l = np.zeros(len(bs), bool)
    for b in high:
        keep_h |= (bs >= iv[b][0]) & (be <= iv[b][1])
    for b in low:
        keep_l |= (bs >= iv[b][0]) & (be <= iv[b][1])
    sel = keep_h | keep_l
    X = np.column_stack([np.ones(sel.sum()), keep_h[sel].astype(float), (bs[sel] - t0) / 3600])
    beta, res, *_ = np.linalg.lstsq(X, rate[sel], rcond=None)
    dof = sel.sum() - 3
    cov = np.linalg.inv(X.T @ X) * (np.sum((rate[sel] - X @ beta) ** 2) / dof)
    p_time = 2 * stats.t.sf(abs(beta[1] / np.sqrt(cov[1, 1])), dof)
    # reading (a) with the serial dependence of the bins kept: the high/low labels of the 1 min bins
    # shifted circularly against the bin rates (vta.circular_shift_p)
    lab = keep_h.astype(int) - keep_l.astype(int)
    _, p_circ, _ = vta.circular_shift_p(lab, rate, lambda l, r: r[l == 1].mean() - r[l == -1].mean(), 10)
    # smallest p that reading (b) can give with these numbers of blocks (complete separation)
    p_b_min = stats.ranksums(np.arange(len(high)) + len(low), np.arange(len(low))).pvalue
    return {"unit": names[i], "rat": rats[i], "blocks_high": len(high), "blocks_low": len(low), "bins_high": len(bh), "bins_low": len(bl),
            "rate_high": rate_h, "rate_low": rate_l, "diff_Hz": rate_h - rate_l,
            "p_a_bins": stats.ranksums(bh, bl).pvalue, "p_b_blocks": stats.ranksums(mh, ml).pvalue,
            "high_coef_with_time_Hz": beta[1], "p_with_time": p_time, "p_a_circular": p_circ, "p_b_smallest_possible": p_b_min,
            "rr_high_minus_low": np.mean([means[b] for b in high]) - np.mean([means[b] for b in low])}
for mb in ("drop", "low"):
    tn = pd.DataFrame([tonic(i, mb) for i in ALL])
    if mb == "drop":
        tonic_drop = tn
    t27 = tn.loc[VTAL]
    print(f"\nMedian block: {mb}")
    if mb == "drop":
        print(tn.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
    for col in ("p_a_bins", "p_b_blocks"):
        k = int((t27[col] < 0.05).sum())
        print(f"  {col}: cells with p < 0.05: {k} of 27 (binomial vs 5%: p = {stats.binomtest(k, 27, 0.05).pvalue:.3g}); "
              f"of 29: {int((tn[col] < 0.05).sum())}")
    kc = int((t27.p_a_circular < 0.05).sum())
    print(f"  variant, reading (a) against circular shifts of the block labels: cells with p < 0.05: {kc} of 27 "
          f"(binomial vs 5%: p = {stats.binomtest(kc, 27, 0.05).pvalue:.3g}); of 29: {int((tn.p_a_circular < 0.05).sum())}")
    nb_low = int((t27.p_b_smallest_possible >= 0.05).sum())
    print(f"  reading (b): cells whose numbers of blocks cannot give p < 0.05 even with complete separation: {nb_low} of 27 "
          f"(blocks high/low per cell: {sorted(zip(t27.blocks_high, t27.blocks_low))})")
    can = t27.p_b_smallest_possible < 0.05
    print(f"  reading (b) among the {int(can.sum())} cells that can reach p < 0.05: {int((t27.p_b_blocks[can] < 0.05).sum())} significant")
    print("  reading (b), smallest possible p by blocks high/low:",
          t27.groupby(["blocks_high", "blocks_low"]).p_b_smallest_possible.agg(["first", "size"]).round(4).to_dict("index"))
    kt = int((t27.p_with_time < 0.05).sum())
    print(f"  variant, 1 min bins regressed on high/low and time in session: cells with p < 0.05: {kt} of 27; "
          f"median coefficient {t27.high_coef_with_time_Hz.median():.3f} Hz")
    w = stats.wilcoxon(t27.rate_high, t27.rate_low)
    print(f"  high - low across the 27 cells: median {t27.diff_Hz.median():.3f} Hz, range {t27.diff_Hz.min():.2f} to {t27.diff_Hz.max():.2f}; "
          f"signed-rank p = {w.pvalue:.3g}; mean reward-rate difference between high and low blocks {t27.rr_high_minus_low.mean():.2f}")

**Result: partially reproduced.** The two readings disagree. Under reading (b), the rank-sum of
block means, 2 of 27 cells differ at p < 0.05, and none when the median block is put with the low
blocks. The 6 cells with 2 blocks a side cannot reach p < 0.05 at all (smallest possible p 0.121)
and the 6 with 3 a side only just can (0.0495), so this reading has little power; among the 21
cells that can reach p < 0.05, 2 do (none with the median block put low; binomial p = 0.394 for 2
of 27 against 5%). Under reading (a), the rank-sum of the 1 min bin rates, 18 of 27 cells
differ (binomial p = 1.2e-17 against 5%), 16 with time in session as a covariate, and 9 (7 with the
median block put low) when the labels are shifted circularly against the bins, which keeps their
serial dependence (binomial p = 4.0e-6). Across the 27 cells firing is higher in the high-reward
blocks by a median 0.50 Hz (range -0.63 to 1.38 Hz; signed-rank p = 0.0006). Tonic firing changes
little, about 0.5 Hz on a mean rate near 7.7 Hz, which fits the paper's description of indifference,
but individual cells do change under the tests on 1 min bins, where the claim says no cell did.
Reading (b) finds no more cells than chance, with little power.

## fig2i-burst-rate-no-change

> There was no overall change in the rate at which dopamine cells fired bursts between higher- and
> lower-reward blocks.

Criterion: a claim of absence. Bursts by the 80/160 ms template; burst rate per cell in high and in
low blocks; readings Wilcoxon signed-rank and paired t-test across cells, p < 0.05.

In [ ]:
def per_block_rates(i, median_block="drop"):
    ts = units[i]["segments"]["RL"]["ts"]
    tr = trials[i]
    high, low, _ = vta.block_split(tr, tau_unit[i], median_block)
    iv = vta.block_intervals(tr)
    b = vta.bursts(ts)
    out = {}
    for label, blocks in (("high", high), ("low", low)):
        dur = sum(iv[k][1] - iv[k][0] for k in blocks)
        nb = sum(vta.counts_in(b, np.array([iv[k][0]]), np.array([iv[k][1]]))[0] for k in blocks)
        out[f"bursts_per_min_{label}"] = nb / dur * 60
        for thr in (0.5, 1.0, 2.0):
            out[f"inactive_{thr:g}s_{label}"] = vta.long_isi_time(ts, [iv[k] for k in blocks], thr) / dur
    whole = [(np.nanmin(tr["LightOn"]), np.nanmax(tr["SideOut"]))]
    for thr in (0.5, 1.0, 2.0):
        out[f"inactive_{thr:g}s_task"] = vta.long_isi_time(ts, whole, thr) / (whole[0][1] - whole[0][0])
    return out
for mb in ("drop", "low"):
    br = pd.DataFrame([per_block_rates(i, mb) for i in ALL], index=names)
    if mb == "drop":
        blockrates = br
    b27 = br.iloc[VTAL]
    d = b27.bursts_per_min_high - b27.bursts_per_min_low
    print(f"Median block {mb}: bursts/min high median {b27.bursts_per_min_high.median():.2f}, low {b27.bursts_per_min_low.median():.2f}; "
          f"difference median {d.median():.3f}, range {d.min():.2f} to {d.max():.2f}; "
          f"Wilcoxon p = {stats.wilcoxon(d).pvalue:.3g}; paired t p = {stats.ttest_rel(b27.bursts_per_min_high, b27.bursts_per_min_low).pvalue:.3g}")
print(blockrates[["bursts_per_min_high", "bursts_per_min_low"]].round(2).to_string())
d29 = blockrates.bursts_per_min_high - blockrates.bursts_per_min_low
print(f"Variant, 29 cells: Wilcoxon p = {stats.wilcoxon(d29).pvalue:.3g}")

**Result: reproduced.** The burst rate does not differ between high- and low-reward blocks: median
difference 1.28 bursts/min (range -12.4 to 17.7), Wilcoxon p = 0.645, paired t p = 0.491 (p = 0.714
and 0.549 with the median block put low; Wilcoxon p = 0.536 with 29 cells).

## fig2i-no-inactive-states

> Dopamine cells never switched between active and inactive states; the proportion of time spent
> inactive (long inter-spike intervals) was very low and did not change between higher- and
> lower-reward blocks.

Criterion: long ISI thresholds 0.5, 1 and 2 s; proportion of bandit-task time in ISIs longer than
the threshold per cell, in high and low blocks. "Very low" is a median below 5% of time; "did not
change" is judged by a Wilcoxon signed-rank and a paired t-test across cells, p < 0.05.

In [ ]:
b27 = blockrates.iloc[VTAL]
rows = []
for thr in ("0.5", "1", "2"):
    task = b27[f"inactive_{thr}s_task"]
    h, l = b27[f"inactive_{thr}s_high"], b27[f"inactive_{thr}s_low"]
    rows.append({"threshold_s": thr, "median_task_%": 100 * task.median(), "max_task_%": 100 * task.max(),
                 "median_high_%": 100 * h.median(), "median_low_%": 100 * l.median(),
                 "median_diff_%": 100 * (h - l).median(), "wilcoxon_p": stats.wilcoxon(h, l).pvalue if (h - l).abs().sum() > 0 else np.nan,
                 "paired_t_p": stats.ttest_rel(h, l).pvalue})
inact = pd.DataFrame(rows)
print(inact.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("\nPer cell, % of bandit time in ISIs > 1 s:", (100 * b27["inactive_1s_task"]).round(2).to_dict())

**Result: reproduced.** The median share of bandit time spent in inter-spike intervals longer than
0.5, 1 and 2 s is 4.7%, 0.13% and 0%, below the 5% taken as very low (the most inactive cell spends
18.1% of the time in intervals over 1 s). The share does not differ between high- and low-reward
blocks at any threshold (Wilcoxon p 0.36 to 0.55, paired t p 0.17 to 0.64).

## retro-tagged-cells-resemble-others

> The two retrogradely tagged VTA-l dopamine cells, infected from NAc core, had firing patterns
> closely resembling the other tagged cells, including no tonic firing change with reward rate.

Criterion: for each of the 2 cells, the per-cell test of fig2efgh finds no significant change, and
its session-wide rate lies within 3.7-12.9 Hz widened by 10% (3.33-14.19 Hz). Their Side-In and
event-aligned responses are shown for comparison.

In [ ]:
r2 = tonic_drop.loc[RETRO, ["unit", "rate_high", "rate_low", "diff_Hz", "p_a_bins", "p_a_circular", "p_b_blocks", "p_b_smallest_possible"]].copy()
r2["rate_bandit"] = rates.loc[RETRO, "rate_bandit"].values
r2["rate_all_segments"] = rates.loc[RETRO, "rate_all_segments"].values
r2["rate_in_range"] = r2.rate_bandit.between(3.33, 14.19) & r2.rate_all_segments.between(3.33, 14.19)
print(r2.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
t = np.arange(vm.TRACE_WIN[0], vm.TRACE_WIN[1] + 1e-9, 0.001)
win = (t > 0) & (t <= 0.25)
resp = pd.DataFrame({"unit": names, "retro": [i in RETRO for i in ALL],
                     "SideIn_rew_peak_Hz": [M[i]["traces"]["SideIn_rew"][win].max() for i in ALL],
                     "SideIn_unrew_0-0.5s_Hz": [M[i]["traces"]["SideIn_unrew"][(t > 0) & (t <= 0.5)].mean() for i in ALL],
                     "LightOn_peak_Hz": [M[i]["traces"]["LightOn"][win].max() for i in ALL],
                     "baseline_Hz": [M[i]["traces"]["LightOn"][(t > -0.5) & (t <= 0)].mean() for i in ALL]})
print("\nEvent responses, retro cells against the range of the 27 VTA-l cells:")
for col in resp.columns[2:]:
    v = resp.loc[VTAL, col]
    print(f"  {col}: VTA-l {v.min():.1f}-{v.max():.1f} (median {v.median():.1f}); retro {resp.loc[RETRO, col].round(1).tolist()}")

**Result: reproduced (resting on an assumption).** Both cells' rates (7.85 and 6.17 Hz in the
bandit task) lie in the widened range of the 27 cells, and their event responses lie within the 27
cells' range (peak after rewarded Side-In 27.8 and 17.6 Hz against 8.4 to 45.8 Hz). Under reading (b)
of the tonic test neither cell differs (p = 0.076 and 0.75), and with 5 or 6 blocks a side that test
could reach p = 0.009 and 0.004. Reading (a) against circular shifts of the labels also finds no
change (p = 0.27 and 0.85). Only the plain rank-sum of 1 min bins, which treats the bins as
independent, finds a change in IM1078180827t14a (0.67 Hz, p = 5.8e-5); the outcome rests on setting
that test aside for this reason.

## fig4ac-dopamine-cue-responses

> Light-On, the Go cue and rewarded Side-In (food hopper click) each evoked a rapid firing increase
> in dopamine cells, seen in the great majority of cells.

Criterion: per cell and event, the time-to-cue-response shuffle test; a cell responds when some
bin in the post-cue window passes (Bonferroni over the bins). Post-cue window 0.5 s or 1 s.
"Great majority" is at least 75% of the 27 cells for each event (partially reproduced when more
than half). A chance reference is the same test at events shifted 5 s earlier (computed below for
the 0.5 s window).

In [ ]:
rows = []
for ev in ("LightOn", "Go", "SideIn_rew"):
    for post in (0.5, 1.0):
        r = [M[i]["cue"][(ev, post)] for i in ALL]
        resp27 = sum(r[i][0] for i in VTAL)
        lat = [r[i][1] for i in VTAL if r[i][0]]
        rows.append({"event": ev, "post_window_s": post, "bins": r[0][3], "responding_27": resp27,
                     "fraction_27": resp27 / 27, "responding_29": sum(x[0] for x in r),
                     "median_first_bin_s": np.median(lat) if lat else np.nan})
cue_counts = pd.DataFrame(rows)
print(cue_counts.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
rng = np.random.default_rng(404)
sham = {}
for ev in ("LightOn", "Go", "SideIn_rew"):
    n = 0
    for i in VTAL:
        tr = trials[i]
        e = tr["SideIn"][tr["outcome"] == 1] if ev == "SideIn_rew" else tr[ev]
        n += vta.cue_response(units[i]["segments"]["RL"]["ts"], e - 5.0, rng, post=0.5, n_shuffles=2000)[0]
    sham[ev] = n
unc = {ev: sum(bool(np.any(M[i]["cue"][(ev, 0.5)][2] < 0.01)) for i in VTAL) for ev in ("LightOn", "Go", "SideIn_rew")}
lat5 = [M[i]["cue"][("LightOn_lat_gt_0.5", 0.5)][0] for i in ALL]
print("Light-On, 0.5 s window, trials with latency over 0.5 s only (no Center-In in the window): responding cells",
      sum(lat5[i] for i in VTAL), "of 27; trials kept per cell (median):",
      int(np.median([np.sum(trials[i]["latency"] > 0.5) for i in VTAL])), "of", int(np.median([len(trials[i]["latency"]) for i in VTAL])))
print("Variant without the correction (some bin p < 0.01 in the 0.5 s window), cells of 27:", unc)
print("Chance reference, events shifted 5 s earlier (0.5 s window, 2,000 shuffles): responding cells", sham)
by_rat = pd.DataFrame({"rat": [rats[i] for i in VTAL], **{ev: [M[i]["cue"][(ev, 0.5)][0] for i in VTAL] for ev in ("LightOn", "Go", "SideIn_rew")}})
print("By rat (0.5 s window):")
print(by_rat.groupby("rat").agg(["sum", "size"]).to_string())

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(14, 3.2), sharey=True)
t = np.arange(vm.TRACE_WIN[0], vm.TRACE_WIN[1] + 1e-9, 0.001)
for a, ev in zip(ax, ("LightOn", "CenterIn", "Go", "SideIn_rew")):
    m = np.nanmean([M[i]["traces"][ev] for i in VTAL], 0)
    a.plot(t, m)
    a.axvline(0, color="k", lw=0.5)
    a.set_title(ev)
    a.set_xlabel("s")
ax[0].set_ylabel("Hz, mean of 27 cells")
plt.tight_layout()
plt.show()
print("Population mean range (Hz):", {ev: (round(np.nanmin(np.nanmean([M[i]['traces'][ev] for i in VTAL], 0)), 1),
                                           round(np.nanmax(np.nanmean([M[i]['traces'][ev] for i in VTAL], 0)), 1))
                                      for ev in ("LightOn", "CenterIn", "Go", "SideIn_rew")})

**Result: partially reproduced (resting on an assumption).** With the 0.5 s post-cue window
(adopted; see the deviations), 27 of 27 cells respond to the Go cue and 24 of 27 to rewarded
Side-In, but 16 of 27 (59%) to Light-On: more than half, less than the 75% taken as a great
majority. With the 1 s window, 13 of 27 (48%) respond to Light-On, 27 to the Go cue and 25 to
Side-In. Without the correction for the number of bins, 22 of 27 respond to Light-On. Using only
trials with latency over 0.5 s, so that no Center-In falls in the window, 11 of 27 respond to
Light-On (fewer trials, and any Center-In response removed), so leaving Center-In out of the
window does not bring the count nearer the 75% bar. No cell responds at events
shifted 5 s earlier.

## fig4a-no-dopamine-firing-ramp-before-approach

> The VTA-l dopamine cell population did not increase firing as rats approached the start port
> (before Center-In) or the food port (before Food-Port-In), where NAc dopamine release increased.

Criterion: ramp angle per cell from the mean firing smoothed with a 50 ms Gaussian, the maximum and
minimum in the 0.5 s before Center-In (long-latency trials, > 2 s) and before Food-Port-In
(rewarded trials), signed positive when the later extremum is higher. Readings: Wilcoxon
signed-rank of the angles against zero and a sign test, one-sided for an increase, p < 0.05.
Reproduced when neither event shows a significant increase.

In [ ]:
ramp = pd.DataFrame({"unit": names, "rat": rats, "n_long_trials": [M[i]["n_long"] for i in ALL],
                     "angle_centerin_long": [M[i]["ramp_centerin_long"] for i in ALL],
                     "angle_foodin": [M[i]["ramp_foodin"] for i in ALL],
                     "change_centerin_Hz": [M[i]["ramp_trace_centerin_long"][-1] - M[i]["ramp_trace_centerin_long"][0] for i in ALL],
                     "change_foodin_Hz": [M[i]["ramp_trace_foodin"][-1] - M[i]["ramp_trace_foodin"][0] for i in ALL]})
print(ramp.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
for label, ids in (("27 VTA-l", VTAL), ("29", ALL)):
    for col in ("angle_centerin_long", "angle_foodin"):
        a = ramp.loc[ids, col]
        npos = int((a > 0).sum())
        print(f"{label} {col}: positive {npos} of {len(a)}; median {a.median():.1f} deg; "
              f"Wilcoxon one-sided (increase) p = {stats.wilcoxon(a, alternative='greater').pvalue:.3g}; "
              f"sign test p = {stats.binomtest(npos, len(a), 0.5, alternative='greater').pvalue:.3g}")
print("Angles are near +-90 degrees because time (s) and rate (Hz) are on their own scales; the sign decides.")

**Result: reproduced.** Before Center-In on long-latency trials the ramp angle is positive in 7 of
27 cells (one-sided Wilcoxon p = 0.991, sign test p = 0.997), and before Food-Port-In in 12 of 27
(p = 0.411 and 0.779); neither shows an increase, under either test and with 29 cells.

## fig4e-light-on-alignment

> Dopamine cell firing increases were locked to Light-On, preferentially on short-latency trials;
> all 25 cells with significant increases after Light-On were better aligned to Light-On than to
> Center-In.

Criterion: significant increase after Light-On by the shuffle test (count expected 25 +- 2.5).
Readings of "better aligned": (a) the peak of the trial-averaged spike density, in the 0.5 s
around each event, is higher aligned on Light-On than on Center-In; (b) the trial-to-trial SD of
the time of the response peak (the per-trial maximum between Light-On and 0.5 s after Center-In,
trials with latency under 5 s) is smaller relative to Light-On than to Center-In. For
"preferentially for short-latency trials": the Light-On response (peak in the 0.5 s after) on
short (< 1 s) against long (> 2 s) latency trials, larger on short trials in most cells, Wilcoxon
signed-rank across cells p < 0.05.

In [ ]:
t = np.arange(vm.TRACE_WIN[0], vm.TRACE_WIN[1] + 1e-9, 0.001)
around = (t >= -0.5) & (t <= 0.5)
rows = []
for i in ALL:
    tr = trials[i]
    tp = M[i]["trial_peak_time"]
    ok = np.isfinite(tp)
    lp = M[i]["lighton_peak"]
    short, long = tr["latency"] < 1.0, tr["latency"] > 2.0
    rows.append({"unit": names[i], "resp_0.5s": M[i]["cue"][("LightOn", 0.5)][0], "resp_1s": M[i]["cue"][("LightOn", 1.0)][0],
                 "peak_LO": M[i]["traces"]["LightOn"][around].max(), "peak_CI": M[i]["traces"]["CenterIn"][around].max(),
                 "peak_LO_sd4": M[i]["traces_sd4"]["LightOn"][around].max(), "peak_CI_sd4": M[i]["traces_sd4"]["CenterIn"][around].max(),
                 "sd_rel_LO": np.std(tp[ok] - tr["LightOn"][ok]), "sd_rel_CI": np.std(tp[ok] - tr["CenterIn"][ok]),
                 "LO_peak_short": lp[short].mean(), "LO_peak_long": lp[long].mean(), "n_short": int(short.sum()), "n_long": int(long.sum())})
al = pd.DataFrame(rows)
al["better_a"] = al.peak_LO > al.peak_CI
al["better_b"] = al.sd_rel_LO < al.sd_rel_CI
al["better_a_sd4"] = al.peak_LO_sd4 > al.peak_CI_sd4
print(al.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
for w in ("resp_0.5s", "resp_1s"):
    r27 = al.loc[VTAL][al.loc[VTAL, w]]
    r29 = al[al[w]]
    print(f"\n{w}: responding cells {len(r27)} of 27 ({len(r29)} of 29); better aligned to Light-On: "
          f"reading (a) {int(r27.better_a.sum())} of {len(r27)}, reading (b) {int(r27.better_b.sum())} of {len(r27)} "
          f"(29-cell set: {int(r29.better_a.sum())}, {int(r29.better_b.sum())} of {len(r29)}); "
          f"reading (a) with the SD 4.5 ms kernel {int(r27.better_a_sd4.sum())} of {len(r27)}")
a27 = al.loc[VTAL]
d = a27.LO_peak_short - a27.LO_peak_long
print(f"\nLight-On peak, short minus long latency trials: larger on short in {int((d > 0).sum())} of 27; "
      f"median {d.median():.2f} Hz; Wilcoxon p = {stats.wilcoxon(d).pvalue:.3g}")

**Result: not reproduced.** 16 of 27 cells pass the shuffle test after Light-On (13 with the 1 s
window, 11 on trials with latency over 0.5 s), against 25 +- 2.5. Of these 16, 15 have a higher
trial-averaged peak aligned on Light-On than on Center-In (reading a; 14 with the SD 4.5 ms kernel)
and 5 a smaller trial-to-trial spread of the response peak relative to Light-On (reading b), so not
every cell is better aligned to Light-On under either reading. The count and the all-cells part of
the criterion fail. The Light-On response is larger on short- than on long-latency trials in 22 of
27 cells (median difference 6.70 Hz, Wilcoxon p = 8.4e-5), as the claim states.

## fig5a-early-trial-firing-not-reward-rate

> During the early part of each trial dopamine cell firing did not depend on reward rate, though
> reward rate affected motivation.

Criterion: a claim of absence. Windows: 0.5 s after Light-On and 0.5 s before Center-In. Tests:
per-cell Pearson correlation of trial firing rate with reward rate, judged by the count of cells
with p < 0.01 against chance (binomial against 1%), and a population Friedman test across reward-rate
terciles, p < 0.05. Reproduced when neither shows a dependence. The set is the 29 cells (as Fig. 5's
2/29); the 27 is a variant. The effect of reward rate on motivation is fig1b.

In [ ]:
def terciles(x):
    q = np.quantile(x, [1 / 3, 2 / 3])
    return np.digitize(x, q)
rows = []
for i in ALL:
    rr = rr_lighton[i]
    tq = terciles(rr)
    rec = {"unit": names[i]}
    for key in ("early_after_lighton", "early_before_centerin"):
        y = M[i][key]
        r, p = stats.pearsonr(rr, y)
        rec[f"r_{key}"], rec[f"p_{key}"] = r, p
        # the same correlation against circular shifts of the reward rate over trials (keeps the serial
        # dependence of both series; vta.circular_shift_p)
        rec[f"p_circ_{key}"] = vta.circular_shift_p(rr, y, lambda a, b: np.corrcoef(a, b)[0, 1], 20)[1]
        for k in range(3):
            rec[f"{key}_t{k + 1}"] = y[tq == k].mean()
    rows.append(rec)
early = pd.DataFrame(rows)
print(early.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
for label, ids in (("29", ALL), ("27 VTA-l", VTAL)):
    e = early.loc[ids]
    for key in ("early_after_lighton", "early_before_centerin"):
        k = int((e[f"p_{key}"] < 0.01).sum())
        kneg = int(((e[f"p_{key}"] < 0.01) & (e[f"r_{key}"] < 0)).sum())
        kc = int((e[f"p_circ_{key}"] < 0.01).sum())
        print(f"{label} {key}: variant against circular shifts of reward rate: cells p < 0.01: {kc} of {len(e)}, "
              f"binomial vs 1% p = {stats.binomtest(kc, len(e), 0.01).pvalue:.3g}")
        fr = stats.friedmanchisquare(*[e[f"{key}_t{j}"] for j in (1, 2, 3)])
        print(f"{label} {key}: cells p < 0.01: {k} ({kneg} negative) of {len(e)}, binomial vs 1% p = {stats.binomtest(k, len(e), 0.01).pvalue:.3g}; "
              f"median r {e[f'r_{key}'].median():.3f}; tercile means (low to high) {[round(e[f'{key}_t{j}'].mean(), 2) for j in (1, 2, 3)]}; "
              f"Friedman p = {fr.pvalue:.3g}")

**Result: not reproduced.** Firing in the 0.5 s after Light-On correlates with reward rate at
p < 0.01 in 8 of 29 cells (7 positive), against 1% expected by chance (binomial p = 3.6e-10), and in
the 0.5 s before Center-In in 6 of 29 (4 positive; binomial p = 3.9e-7). The population Friedman
test across terciles finds no dependence (p = 0.080 and 0.19), and the tercile means differ by
less than 0.4 Hz (8.53, 8.76, 8.92 Hz after Light-On). The criterion asks that neither test find a
dependence; the per-cell count does. Against circular shifts of reward rate over trials, which keep
the serial dependence that the Pearson p-values ignore, only 1 and 2 of 29 cells are significant
(binomial p = 0.25 and 0.034), so most of the per-cell count may come from slow drift shared by
firing and reward rate.

## fig5a-positive-rpe-reward-cue

> The phasic dopamine response to the reward cue at Side-In was reliably stronger when reward rate
> was lower, consistent with positive RPE coding, strongly and consistently across cells.

Criterion: peak firing per rewarded trial (maximum of the spike density in the 250 ms after
Side-In), correlated per cell with reward rate (Pearson; Spearman a variant), significant at
p < 0.01 before correction and after Bonferroni over 29 cells. Reproduced when the population mean
peak is highest in the lowest reward-rate tercile and lowest in the highest, the median correlation
is negative (Wilcoxon signed-rank across cells, p < 0.05), and significant negative correlations
outnumber positive ones. The set is the 29 cells; the 27 is a variant. The kernel's literal reading
(SD 4.5 ms) is a variant.

In [ ]:
def corr_table(estimate, measure, outcome, method="pearson"):
    rows = []
    for i in ALL:
        sel = trials[i]["outcome"] == outcome
        x, y = estimate[i][sel], M[i][measure][sel]
        ok = np.isfinite(x) & np.isfinite(y)
        x, y = x[ok], y[ok]
        r, p = (stats.pearsonr(x, y) if method == "pearson" else stats.spearmanr(x, y))
        tq = terciles(x)
        rows.append({"unit": names[i], "n": len(x), "r": r, "p": p, "t1": y[tq == 0].mean(), "t2": y[tq == 1].mean(), "t3": y[tq == 2].mean()})
    return pd.DataFrame(rows)

def positive_summary(tab, ids, label):
    t = tab.loc[ids]
    n = len(ALL)
    neg_b, pos_b = int(((t.p < 0.01) & (t.r < 0)).sum()), int(((t.p < 0.01) & (t.r > 0)).sum())
    neg_a, pos_a = int(((t.p < 0.01 / n) & (t.r < 0)).sum()), int(((t.p < 0.01 / n) & (t.r > 0)).sum())
    w = stats.wilcoxon(t.r)
    terc = [t.t1.mean(), t.t2.mean(), t.t3.mean()]
    ok = terc[0] == max(terc) and terc[2] == min(terc) and t.r.median() < 0 and w.pvalue < 0.05 and neg_b > pos_b
    print(f"{label}: tercile means (low to high rate) {[round(v, 1) for v in terc]} Hz; median r {t.r.median():.3f} "
          f"(Wilcoxon p = {w.pvalue:.2g}); significant before correction neg {neg_b} / pos {pos_b}; after Bonferroni over {n}: "
          f"neg {neg_a} / pos {pos_a}; criterion met: {ok}")
    return ok, neg_b, neg_a

pos = corr_table(rr_sidein, "peak_sd20", 1)
print(pos.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
pos_res = positive_summary(pos, ALL, "29 cells, SD 20 ms, Pearson")
positive_summary(pos, VTAL, "27 cells, SD 20 ms, Pearson")
positive_summary(corr_table(rr_sidein, "peak_sd20", 1, "spearman"), ALL, "29 cells, SD 20 ms, Spearman")
positive_summary(corr_table(rr_sidein, "peak_sd4", 1), ALL, "29 cells, SD 4.5 ms, Pearson")
print("By rat, median r:", pos.assign(rat=rats).groupby("rat").r.median().round(3).to_dict())

In [ ]:
tercile_traces = []
for i in ALL:
    ts = units[i]["segments"]["RL"]["ts"]
    s20 = vta.sdf(ts, ts[-1] + 5)
    d = {}
    for outcome in (1, 0):
        sel = trials[i]["outcome"] == outcome
        tq = terciles(rr_sidein[i][sel])
        for k in range(3):
            d[(outcome, k)] = np.nanmean(vta.aligned(s20, trials[i]["SideIn"][sel][tq == k], *vm.TRACE_WIN), 0)
    tercile_traces.append(d)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
t = np.arange(vm.TRACE_WIN[0], vm.TRACE_WIN[1] + 1e-9, 0.001)
for a, outcome, title in ((ax[0], 1, "rewarded"), (ax[1], 0, "unrewarded")):
    for k, col in enumerate(("tab:red", "tab:orange", "tab:green")):
        a.plot(t, np.nanmean([tercile_traces[i][(outcome, k)] for i in ALL], 0), color=col, label=["low", "middle", "high"][k] + " reward rate")
    a.set_title(f"Side-In, {title}")
    a.set_xlabel("s")
ax[0].set_ylabel("Hz, mean of 29 cells")
ax[0].legend()
plt.tight_layout()
plt.show()
for outcome in (1, 0):
    m = [np.nanmean([tercile_traces[i][(outcome, k)] for i in ALL], 0) for k in range(3)]
    print(f"{'rewarded' if outcome else 'unrewarded'}: range of the tercile means {min(np.nanmin(x) for x in m):.1f}-{max(np.nanmax(x) for x in m):.1f} Hz")

**Result: reproduced.** The peak after rewarded Side-In falls with reward rate: tercile means
38.2, 31.4 and 28.2 Hz from low to high; median r -0.310 (Wilcoxon p = 3.7e-8); 21 cells with a
significant negative correlation and none positive before correction, 17 and 0 after. The result
holds with Spearman's rho (20 negative), with the 27 cells (20) and with the SD 4.5 ms kernel (13
negative, 0 positive).

## fig5ab-weak-negative-rpe

> When the reward cue was omitted dopamine cells paused, but encoding of negative RPEs was much
> weaker or absent, at the population level and in individual cells; negative RPE coding by pause
> duration was seen in only 2 of 29 neurons.

Criterion: on unrewarded trials, the valley (minimum smoothed firing) and the pause (maximum ISI
within 2 s after Side-In), each correlated per cell with reward rate; negative RPE coding is a
lower valley or a longer pause at higher reward rate. Valley windows: 1 to 3 s after Side-In
(Methods) and 0 to 2 s (Extended Data Fig. 8 legend). Count of significant pause cells before
(p < 0.01) and after correction. Reproduced when the pause count matches 2 (tolerance 0.2) under a
reading and fewer cells have significant valley correlations in the negative-RPE direction than
have significant peak correlations in fig5a-positive-rpe-reward-cue. How the pause is measured at
the window's edges is not given; three readings are run: the window bounds count as spike
boundaries (`pause_edges`), only ISIs between spikes inside the window (`pause_isi`), and whole ISIs
of the spike train that overlap the window (`pause_span`).

In [ ]:
def negative_summary(estimate, label, peak_counts, sd="sd20"):
    n = len(ALL)
    out = {"estimate": label}
    for key in vm.VALLEY_WINS:
        t = corr_table(estimate, f"valley_{key}_{sd}", 0)
        out[f"valley_{key}_neg_before"] = int(((t.p < 0.01) & (t.r < 0)).sum())
        out[f"valley_{key}_neg_after"] = int(((t.p < 0.01 / n) & (t.r < 0)).sum())
        out[f"valley_{key}_pos_before"] = int(((t.p < 0.01) & (t.r > 0)).sum())
        out[f"valley_{key}_median_r"] = t.r.median()
    for key in ("pause_edges", "pause_isi", "pause_span"):
        t = corr_table(estimate, key, 0)
        out[f"{key}_pos_before"] = int(((t.p < 0.01) & (t.r > 0)).sum())
        out[f"{key}_pos_after"] = int(((t.p < 0.01 / n) & (t.r > 0)).sum())
        out[f"{key}_neg_before"] = int(((t.p < 0.01) & (t.r < 0)).sum())
        out[f"{key}_median_r"] = t.r.median()
    out["peak_neg_before"], out["peak_neg_after"] = peak_counts
    return out

neg = negative_summary(rr_sidein, "leaky reward rate", pos_res[1:])
print(pd.Series(neg).to_string())
pause_counts = {k: (neg[f"{k}_pos_before"], neg[f"{k}_pos_after"]) for k in ("pause_edges", "pause_isi", "pause_span")}
valley_vs_peak = all(neg[f"valley_{k}_neg_before"] < neg["peak_neg_before"] for k in vm.VALLEY_WINS)
print("\nPause counts (before, after correction) under the three readings:", pause_counts, "; expected 2")
n_nan = {names[i]: int(np.isnan(M[i]["pause_isi"][trials[i]["outcome"] == 0]).sum()) for i in ALL}
print("Unrewarded trials dropped by the ISI-only reading (fewer than 2 spikes in the 2 s), per cell:",
      {k: v for k, v in n_nan.items() if v}, "; all other cells 0; total", sum(n_nan.values()), "of",
      sum(int((trials[i]["outcome"] == 0).sum()) for i in ALL))
print("Valley significant (negative-RPE direction) fewer than peak significant, both windows:", valley_vs_peak)
print("Variant, SD 4.5 ms valleys:", {k: v for k, v in negative_summary(rr_sidein, "sd4", pos_res[1:], "sd4").items() if k.startswith("valley") and "after" not in k})
vt = corr_table(rr_sidein, "valley_methods_1to3s_sd20", 0)
print(f"Population, unrewarded valley (1-3 s) by tercile (low to high rate): {[round(vt[c].mean(), 2) for c in ('t1', 't2', 't3')]} Hz; "
      f"Friedman p = {stats.friedmanchisquare(vt.t1, vt.t2, vt.t3).pvalue:.3g}")
for key in vm.VALLEY_WINS:
    v = np.concatenate([M[i][f"valley_{key}_sd20"][trials[i]["outcome"] == 0] for i in ALL])
    vc = corr_table(rr_sidein, f"valley_{key}_sd20", 0)
    print(f"Valley {key}: unrewarded trials with a minimum under 0.5 Hz: {int(np.sum(v < 0.5))} of {len(v)} (largest minimum {np.nanmax(v):.3g} Hz); "
          f"cells with a defined correlation: {int(vc.r.notna().sum())}, of them p < 0.01: {int((vc.p < 0.01).sum())}; "
          f"cells whose valley is the same on every unrewarded trial (correlation undefined): "
          f"{sum(np.ptp(M[i][f'valley_{key}_sd20'][trials[i]['outcome'] == 0]) < 1e-6 for i in ALL)} of 29 (SD 20 ms), "
          f"{sum(np.ptp(M[i][f'valley_{key}_sd4'][trials[i]['outcome'] == 0]) < 1e-6 for i in ALL)} of 29 (SD 4.5 ms)")
pt = corr_table(rr_sidein, "pause_edges", 0)
print(f"Population, pause (edges) by tercile: {[round(pt[c].mean(), 3) for c in ('t1', 't2', 't3')]} s; "
      f"Friedman p = {stats.friedmanchisquare(pt.t1, pt.t2, pt.t3).pvalue:.3g}")
print("Median unrewarded pause (s) per cell, range:", round(pt[["t1", "t2", "t3"]].mean(axis=1).min(), 2), "-", round(pt[["t1", "t2", "t3"]].mean(axis=1).max(), 2))

In [ ]:
# The authors' scripts in the data set: what they compute for the valley, the pause and the kernel
code = {n: vta.read_member(f"data_and_code/{n}").decode("latin1") for n in ("Fig5_replicate.m", "getFiringRate.m", "pelletpermin.m")}
for n, txt in code.items():
    hits = [l.strip() for l in txt.splitlines() if any(w in l.lower() for w in ("min(", "isi", "pause", "valley"))]
    print(f"{n}: lines with min(, isi, pause or valley:", hits or "none")
print("getFiringRate.m, Gaussian case:", [l.strip() for l in code["getFiringRate.m"].splitlines() if "sigma = twsize" in l or "normpdf" in l])
print("Fig5_replicate.m, kernel and call:", [l.strip() for l in code["Fig5_replicate.m"].splitlines() if l.startswith("twsize") or "getFiringRate(" in l])

**Result: partially reproduced.** The count of pause cells depends on how the pause is measured:
2 of 29 cells have a pause that lengthens significantly with reward rate, before and after
correction, when only intervals between spikes inside the 2 s window count (this drops 50 of 3695
trials, 38 of them in one cell); 8 before and 5 after correction when the window's bounds count; and
9 and 5 with whole inter-spike intervals that overlap the window. Only the first matches 2. Under
every reading negative coding is weaker than positive coding (at most 9 pause cells against 21 with
significant peak correlations). The valley part of the criterion cannot be tested: the minimum of
the spike density is under 0.5 Hz on 3694 of the 3695 unrewarded trials in the 1 to 3 s window
(the largest is 0.91 Hz) and on all 3695 in the 0 to 2 s window. It is the same on every trial in
21 and 27 of the 29 cells (the two windows, SD 20 ms) and in all 29 with the SD 4.5 ms kernel, so
a correlation is defined only for 8 and 2 cells, and none of these is significant. With the valley
at the floor on nearly every trial, the criterion's fewer valley than peak cells holds by
construction. The authors' `Fig5_replicate.m` computes neither the valley
nor the pause. At the population level the pause lengthens slightly with reward rate under the
bounds reading (0.49, 0.52, 0.55 s by tercile, Friedman p = 9.7e-5).

## ed8-other-reward-expectation-estimates

> Similar dopamine firing results held when reward expectation was estimated by trial-based
> actor-critic or Q-learning models or by counting recent rewards.

Criterion: reward expectation as the rewards in the previous 10 trials; the actor-critic V; the
Q-learning sum Q_left + Q_right; alpha per session by the most negative correlation with log
latency (0.01 to 1 in steps of 0.01), initial values 0.5. Reproduced when, under each estimate, the
criteria of fig5a-positive-rpe-reward-cue hold and negative RPE coding is weaker than positive as
in fig5ab-weak-negative-rpe (fewer significant valley or pause cells than significant peak cells).

In [ ]:
alphas = {}
for s in sessions:
    tr = trials[first_unit[s]]
    alphas[s] = {"ac": vta.best_alpha(tr, "ac"), "q": vta.best_alpha(tr, "q")}
print(pd.DataFrame({s: {f"{m}_alpha": a[m][0] for m in a} | {f"{m}_r": round(a[m][1], 3) for m in a} for s, a in alphas.items()}).T.to_string())
for m in ("ac", "q"):
    print(f"{m}: sessions with alpha at the grid's edge, 1.00: {sum(a[m][0] == 1.0 for a in alphas.values())}, "
          f"0.01: {sum(a[m][0] == 0.01 for a in alphas.values())} of {len(alphas)}")
estimates = {
    "rewards in last 10 trials": [vta.rewards_last10(trials[i]).astype(float) for i in ALL],
    "actor-critic V": [vta.trial_learning_values(trials[i], alphas[units[i]["session"]]["ac"][0], "ac") for i in ALL],
    "Q-learning Q_left + Q_right": [vta.trial_learning_values(trials[i], alphas[units[i]["session"]]["q"][0], "q") for i in ALL],
}
rows = []
for label, est in estimates.items():
    tab = corr_table(est, "peak_sd20", 1)
    ok, nb, na = positive_summary(tab, ALL, label)
    ng = negative_summary(est, label, (nb, na))
    weaker = all(ng[f"valley_{k}_neg_before"] < nb for k in vm.VALLEY_WINS) and max(ng["pause_edges_pos_before"], ng["pause_isi_pos_before"], ng["pause_span_pos_before"]) < nb
    rows.append({**ng, "positive_criteria_met": ok, "negative_weaker": weaker})
ed8 = pd.DataFrame(rows).set_index("estimate").T
print(ed8.to_string())

**Result: partially reproduced.** With the rewards in the last 10 trials and with the actor-critic
V, every criterion of fig5a-positive-rpe-reward-cue holds (20 and 23 cells significantly negative,
none positive). With the Q-learning sum the peak still falls with expectation (median r -0.215,
Wilcoxon p = 1.1e-8, 16 cells negative, none positive) but the middle tercile (29.8 Hz) is below the
highest (30.1 Hz), so the tercile part of the criterion fails for that estimate. Negative coding is
weaker than positive under all three estimates by the pause (2 pause cells under the interval
reading, 8, 6 and 4 under the bounds reading, 6 under the whole-ISI reading, against 20, 23 and 16
peak cells); the valley gives no significant cell, but as in fig5ab-weak-negative-rpe it is
degenerate and cannot show a difference. The fitted alpha is at the grid's edge (1.00) in 2
sessions for the actor-critic and 4 for Q-learning (0.01 in 1). In two of these sessions,
IM-1003_170404 and IM-1037_180112, the latency fit is weak under both models (r -0.165 and -0.035
for the actor-critic, -0.167 and 0.016 for Q-learning); the other two Q-learning sessions at 1.00
fit well (IM-1002_170127, r -0.346; IM-1078_180827, r -0.402).

## Summary

| Claim | Outcome |
|---|---|
| fig1b-latency-scales-with-reward-rate | reproduced |
| fig2-tagged-dopamine-cells | partially reproduced |
| fig2d-dopamine-firing-rates | reproduced |
| ed5-pavlovian-dopamine-click-responses | partially reproduced |
| ed5-pavlovian-dopamine-cue-responses | partially reproduced |
| ed5a-pavlovian-behavior | reproduced |
| fig2efgh-tonic-firing-indifferent-to-reward-rate | partially reproduced |
| fig2i-burst-rate-no-change | reproduced |
| fig2i-no-inactive-states | reproduced |
| retro-tagged-cells-resemble-others | reproduced (assumed test) |
| fig4ac-dopamine-cue-responses | partially reproduced (assumed window) |
| fig4a-no-dopamine-firing-ramp-before-approach | reproduced |
| fig4e-light-on-alignment | not reproduced |
| fig5a-early-trial-firing-not-reward-rate | not reproduced |
| fig5a-positive-rpe-reward-cue | reproduced |
| fig5ab-weak-negative-rpe | partially reproduced |
| ed8-other-reward-expectation-estimates | partially reproduced |
| 20 claims on microdialysis, photometry, in vitro imaging, histology and untagged VTA-l units | blocked, data_missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The legends of main Figures 1 to 5 were not in the text
the claims were written from, so several tests and windows are readings named in the claims file
or taken from the Methods: the per-cell test of tonic firing (both readings and a circular-shift variant
are run; for the retrograde cells the tests that allow for serial dependence are adopted), the
post-cue window of the shuffle test (0.5 s, adopted), and the pause measure (three readings run). The Gaussian kernel's SD (20 ms) is taken from the authors' code in the data set.
The selection of tagged cells cannot be repeated, as only the selected cells are deposited.

What was found about the data set. It holds only the 29 tagged dopamine cells, not the paper's
microdialysis, photometry, histology, in vitro or untagged unit data. The descriptor says the cells
come from 4 rats; they come from 5. The `CNIlatency` field is clipped at 65535 ms. Spike times are
out of order in 11 units. The data hold 8 IM-1037 cells where the Methods give 9 (whose per-rat
counts sum to 28, not the stated 27), and 8 cells with 75/25% Pavlovian tones where Extended Data
Fig. 5 gives 9. The meaning of the `FR` field, the alignment of the food-port status and the names
of the event rows are not documented, and the light-evoked waveforms of 4 cells correlate below 0.9
with the session-wide ones on every reading, where the Methods say all exceed 0.9.

In [ ]:
print(f"Notebook run time: {(time.time() - started) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024:.0f} MB")